# Hugging Face Pipeline Applications

1. Execute a atividade em grupo
2. Postar a atividade [aqui](https://drive.google.com/drive/folders/1LRpP7Vqs4nSa1fxwAphysI6AHjURnwIF?usp=sharing)
3. Empregue somente modelos abertos



### Alunos

1. Daianny Quieroz
2. Rodrigo Madera
3. Sara Pires
4. Tiago Carvalho


...


# 1. Task: Chat com contexto (`Text-Generation`)

Empregando ao menos um modelo diferente do empregado em sala, implemente e compare o resultado de dois modelos de Chat com contexto. Discuta em poucas linhas o resultado. Chat em português.

In [ ]:
# seus modelos

import torch
import gc

if 'chat' in globals():
    del chat
if 'pipe' in globals():
    del pipe

# 2. Chamar o coletor de lixo do Python para limpar a memória RAM
gc.collect()

# 3. Limpar o cache da GPU no PyTorch
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print("Memória da GPU liberada com sucesso!")
else:
    print("GPU não disponível ou já liberada.")

Memória da GPU liberada com sucesso!


In [ ]:
!pip install -q transformers accelerate sentencepiece protobuf

In [ ]:

import torch
import transformers
from transformers import pipeline

print("Versão do transformers:", transformers.__version__)
print("GPU disponível:", torch.cuda.is_available())

# Dispositivo: 0 = primeira GPU, -1 = CPU
DEVICE = 0 if torch.cuda.is_available() else -1
# Em GPU usamos meia precisão (float16) para economizar memória
DTYPE = torch.float16 if torch.cuda.is_available() else torch.float32

Versão do transformers: 5.22.1
GPU disponível: True


In [ ]:

#Modelo do Chat

MODELO_CHAT = "Qwen/Qwen3.5-0.8B"

chat = pipeline(
    "text-generation",
    model=MODELO_CHAT,
    dtype=DTYPE,
    device=DEVICE,
)

config.json:   0%|          | 0.00/2.91k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/50.9k [00:00<?, ?B/s]

model.safetensors-00001-of-00001.safeten(…): reconstructing file:   0%|          |  0.00B / 1.75GB            

model.safetensors-00001-of-00001.safeten(…): downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/16.7k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 12.8MB            

tokenizer.json: downloading bytes:           |  0.00B            

chat_template.jinja:   0%|          | 0.00/7.75k [00:00<?, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/390 [00:00<?, ?B/s]

## **Discussão**

Observamos que ao modificar as variáveis "do_sample" e "temperature" a resposta, ou seja, a saída do texto sempre é modificada para a abordagem do usuário "Explique em duas frases o que é um modelo de linguagem."

# 2. Task: `Feature-Extration`

Empregue uma task `Feature-Extration` para criar o *embbeding* de duas frases semelhantes em português. Mostre os tokens associados à cada frase e verifique a similaridade dos embbedings das duas frases. Discuta o seu resultado.

<br>

> Opcional: empregue

> ```
> from sklearn.manifold import TSNE
> tsne = TSNE(n_components=2, perplexity=3, random_state=42, n_iter=1000)
> vetores_2d = tsne.fit_transform(todos_os_vetores)
> ```

> para comparar os vetores gerados.

In [ ]:

!pip install -q transformers sentence-transformers torch scikit-learn

import torch
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics.pairwise import cosine_similarity

# Modelo multilíngue para geração de embeddings
MODELO = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

tokenizer = AutoTokenizer.from_pretrained(MODELO)
model = AutoModel.from_pretrained(MODELO)

#  frases semelhantes em português
frases = [
    "A inteligência artificial pode auxiliar na análise de dados.",
    "A IA pode ajudar no processamento e na análise das informações."
]


tokens = tokenizer(
    frases,
    padding=True,
    truncation=True,
    return_tensors="pt"
)

print("TOKENS DAS FRASES:\n")

for i, frase in enumerate(frases):
    print(f"Frase {i+1}: {frase}")
    print("Tokens:", tokenizer.tokenize(frase))
    print()

# Geração dos embeddings
with torch.no_grad():
    saida = model(**tokens)

# Mean Pooling: média dos vetores dos tokens válidos
mascara = tokens["attention_mask"].unsqueeze(-1)

embeddings = (
    saida.last_hidden_state * mascara
).sum(dim=1) / mascara.sum(dim=1)

# Normalização dos vetores
embeddings = torch.nn.functional.normalize(
    embeddings, p=2, dim=1
)

# Similaridade de cosseno
similaridade = cosine_similarity(
    embeddings.cpu().numpy()
)

print("Similaridade entre as frases:")
print(f"{similaridade[0][1]:.4f}")

print("\nDimensão dos embeddings:")
print(embeddings.shape)


config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 9.08MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  471MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

TOKENS DAS FRASES:

Frase 1: A inteligência artificial pode auxiliar na análise de dados.
Tokens: ['▁A', '▁inteligência', '▁artificial', '▁pode', '▁auxiliar', '▁na', '▁análise', '▁de', '▁dados', '.']

Frase 2: A IA pode ajudar no processamento e na análise das informações.
Tokens: ['▁A', '▁I', 'A', '▁pode', '▁ajudar', '▁no', '▁process', 'amento', '▁e', '▁na', '▁análise', '▁das', '▁informações', '.']

Similaridade entre as frases:
0.8078

Dimensão dos embeddings:
torch.Size([2, 384])


## **Discussão**

O modelo transformou as duas frases em números que representam seus significados. Mesmo escritas de formas diferentes, elas apresentam ideias semelhantes.
O resultado da comparação mostra o quanto as frases estão relacionadas. Esse recurso pode ser utilizado para encontrar textos com assuntos parecidos e facilitar pesquisas de informações.


# 3. Task: `Table-Question-Answer`

Escolha uma tabelas de dados aberta de sua preferência e explore duas ou três questões sobre esses dados empregando um modelo de  `Table-Question-Answer`. Discuta os resultados.

In [ ]:

!pip install -q transformers torch pandas requests

import pandas as pd
import requests
from transformers import pipeline


# Consulta aos dados do IBGE - Censo 2022


URL = "https://apisidra.ibge.gov.br/values/t/4709/n6/all/v/93/p/2022"

resposta = requests.get(URL, timeout=120)
resposta.raise_for_status()

dados = resposta.json()

# Os dados da API possuem campos identificados por códigos.
# D1C = Código do município
# D1N = Nome do município
# V   = Valor da população

# Códigos IBGE dos municípios selecionados
codigos = [
    "3525904",  # Jundiaí
    "3556503",  # Várzea Paulista
    "3509502",  # Campinas
    "3548500",  # Santos
    "3550308"   # São Paulo
]

# Criar tabela utilizando diretamente os campos da API
registros = []

for linha in dados[1:]:

    codigo = linha["D1C"]

    if codigo in codigos:

        registros.append({
            "Municipio": linha["D1N"],
            "Populacao": linha["V"]
        })

tabela = pd.DataFrame(registros)

# Conferir se os municípios foram encontrados
if tabela.empty:
    raise ValueError("Nenhum município foi encontrado na consulta do IBGE.")

# Converter população para número
tabela["Populacao"] = pd.to_numeric(
    tabela["Populacao"]
)

# Ordenar pela população
tabela = tabela.sort_values(
    by="Populacao",
    ascending=False
).reset_index(drop=True)

# Converter os valores para texto para utilização no TAPAS
tabela["Populacao"] = tabela["Populacao"].astype(str)

print("Dados do Censo Demográfico 2022:")
display(tabela)


#  Carregar o modelo TAPAS


MODELO_TQA = "google/tapas-base-finetuned-wtq"

qa = pipeline(
    "table-question-answering",
    model=MODELO_TQA
)


#  Perguntas sobre a tabela


perguntas = [
    "Which municipality has the largest population?",
    "What is the population of Jundiaí - SP?",
    "Which municipality has the smallest population?"
]


#  Executar as perguntas


for pergunta in perguntas:

    resultado = qa(
        table=tabela,
        query=pergunta
    )

    print("\nPergunta:", pergunta)
    print("Resposta:", resultado["answer"])


Dados do Censo Demográfico 2022:


,Municipio,Populacao
0,São Paulo - SP,11451999
1,Campinas - SP,1139047
2,Jundiaí - SP,443221
3,Santos - SP,418608
4,Várzea Paulista - SP,115771


config.json:   0%|          | 0.00/1.66k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  443MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

... (2 outputs left out)

model.safetensors: reconstructing file:   0%|          |  0.00B /  443MB            

model.safetensors: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/154 [00:00<?, ?B/s]


Pergunta: Which municipality has the largest population?
Resposta: São Paulo - SP

Pergunta: What is the population of Jundiaí - SP?
Resposta: AVERAGE > 443221

Pergunta: Which municipality has the smallest population?
Resposta: Várzea Paulista - SP


## **Discussão**

O modelo foi utilizado para responder perguntas diretamente sobre os dados do Censo 2022. Com isso, conseguimos consultar informações sobre a população de diferentes municípios sem precisar procurar manualmente na tabela.
A atividade demonstra como a inteligência artificial pode facilitar a consulta de dados e ajudar na obtenção de informações de maneira mais rápida.

# 4. Task: uma de sua preferência  

 Implmente uma taks de sua preferência não empregada sem sala de aula. Discuta os resultados.

 Sugestão: **Reconhecimento de Entidades Nomeadas (NER)**

In [ ]:

!pip install -q transformers torch

from transformers import pipeline

# Modelo de reconhecimento de entidades em português
MODELO = "pierreguillou/ner-bert-large-cased-pt-lenerbr"

ner = pipeline(
    "ner",
    model=MODELO,
    aggregation_strategy="simple"
)

# Texto para análise
texto = """
Maria Silva trabalha na Universidade de São Paulo,
localizada em São Paulo. Recentemente, participou de
uma reunião com representantes do Ministério da Saúde
em Brasília para discutir projetos de tecnologia.
"""

# Aplicação do reconhecimento de entidades
entidades = ner(texto)

print("ENTIDADES IDENTIFICADAS:\n")

for entidade in entidades:

    print(
        "Entidade:", entidade["word"],
        "| Categoria:", entidade["entity_group"],
        "| Confiança:", round(entidade["score"], 4)
    )


config.json:   0%|          | 0.00/1.54k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 1.33GB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/562 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.33GB            

model.safetensors: downloading bytes:           |  0.00B            

vocab.txt:   0%|          | 0.00/210k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/438k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

ENTIDADES IDENTIFICADAS:

Entidade: Maria Silva | Categoria: PESSOA | Confiança: 0.9754
Entidade: Universidade de São Paulo | Categoria: ORGANIZACAO | Confiança: 0.9987
Entidade: São Paulo | Categoria: LOCAL | Confiança: 0.9961
Entidade: Ministério da Saúde | Categoria: ORGANIZACAO | Confiança: 0.9996
Entidade: Brasília | Categoria: LOCAL | Confiança: 0.9992


## **Discussão**

O modelo conseguiu identificar informações importantes em um texto, como nomes de pessoas, instituições e cidades.
Esse recurso pode facilitar a organização de documentos e a localização de informações importantes, principalmente quando trabalhamos com grandes quantidades de textos.

# 5. Next  

Discuta com o grupo e resuma uma ideia de solução que vocês implementariam com o que aprendeu desses modelos. Procure explorar algo do seu trabalho, ou relacionado ao seu tema de pesquisa. Busque uma implementação o mais original possível.



## **Discussão**

Tema: Sistema simples para organização e consulta de doações de alimentos em uma instituição social.
Ideia da solução
Propomos desenvolver uma ferramenta para auxiliar uma igreja ou organização social a organizar as doações de alimentos recebidas e distribuídas às famílias atendidas.

Atualmente, esse controle pode ser feito por planilhas ou anotações, dificultando a consulta sobre quais alimentos estão disponíveis e quais precisam ser repostos.
A proposta é criar uma aplicação simples na qual os responsáveis possam cadastrar as doações e consultar as informações utilizando perguntas em linguagem natural.
Como funcionaria?
- Cadastro: registrar os alimentos recebidos, suas quantidades e datas.
- Consulta inteligente: permitir perguntas como : "Quantas cestas básicas temos disponíveis?" ou "Quais alimentos estão em falta?".
- Resumo automático: gerar um resumo das doações recebidas e distribuídas.
Modelos que poderiam ser utilizados:

Modelo	Aplicação
Google TAPAS	Responder perguntas sobre os dados cadastrados em uma tabela.
Qwen	Interpretar perguntas e gerar respostas em português.

Relevância da solução
A ferramenta ajudaria organizações sociais a controlar melhor as doações, consultar os estoques e organizar a distribuição dos alimentos.
É uma proposta simples, que pode ser desenvolvida utilizando uma planilha como base de dados, sem depender de informações de órgãos públicos ou integrações complexas.
A ideia é aplicar os conhecimentos adquiridos na disciplina para desenvolver uma solução que possa ser utilizada na prática por uma organização social.